# 01: Translation run

The **only** notebook that calls the LLM. Each matrix cell (target x model) has its own code cell below, so one (target, model) pair can be run or re-run in isolation and its per-query results inspected immediately. Records go to `records_<dataset>_<target>_<model>.json` in `eval/outputs/records/`; downstream notebooks glob those files.

Configuration lives in `harness.config` (temperature 0.0, max 3 iterations, offline syntax validation). Token counts come straight from `result.token_usage` -- no log scraping. Resume is automatic: query ids already on disk for a cell are skipped; delete a cell's records file to force a full re-run. Run the cells serially, top to bottom; never run model cells in parallel.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "eval"))

from dataclasses import replace

import pandas as pd

from harness import RECORDS_DIR, RUN_MATRIX, billed_input_tokens, load_records, run_translation

print(f'{len(RUN_MATRIX)} matrix cell(s):')
for rc in RUN_MATRIX:
    print(f'  - {rc.dataset} x {rc.target} x {rc.model} ({rc.provider})')

12 matrix cell(s):
  - ldbc x cypher x llama3.2:latest (ollama)
  - ldbc x cypher x qwen3-coder:30b (ollama)
  - ldbc x cypher x gemma4:26b (ollama)
  - ldbc x cypher x claude-opus-4-8 (anthropic)
  - ldbc x aql x llama3.2:latest (ollama)
  - ldbc x aql x qwen3-coder:30b (ollama)
  - ldbc x aql x gemma4:26b (ollama)
  - ldbc x aql x claude-opus-4-8 (anthropic)
  - ldbc x gremlin x llama3.2:latest (ollama)
  - ldbc x gremlin x qwen3-coder:30b (ollama)
  - ldbc x gremlin x gemma4:26b (ollama)
  - ldbc x gremlin x claude-opus-4-8 (anthropic)


## Helpers and smoke subset

`run_cell(target, model)` runs one matrix cell and returns its per-query results table; `summarize_target(target)` aggregates one target's records on disk by model. Set `SUBSET` to a tuple of query ids (e.g. `('ldbc_q01',)`) to restrict every cell to a smoke subset before paying for a full run; `None` runs the whole gold set.

In [2]:
SUBSET: tuple[str, ...] | None = None  # e.g. ('ldbc_q01',) for a smoke run

RESULT_COLS = ['query_id', 'difficulty', 'validation_passed', 'iterations_used', 'status',
               'billed_input_tokens', 'output_tokens', 'duration_seconds', 'error']

def find_cell(target, model):
    """The unique RUN_MATRIX cell for (target, model)."""
    matches = [rc for rc in RUN_MATRIX if rc.target == target and rc.model == model]
    assert len(matches) == 1, f'{len(matches)} matrix cell(s) for {target}/{model}, expected 1'
    return matches[0]

def run_cell(target, model):
    """Run one matrix cell (serially); returns its per-query results table."""
    rc = find_cell(target, model)
    if SUBSET is not None:
        rc = replace(rc, subset=SUBSET)
    df = pd.DataFrame(run_translation(rc))
    if not len(df):
        print('No records.')
        return df
    df['billed_input_tokens'] = billed_input_tokens(df['input_tokens'], df['cache_read_tokens'], df['cache_creation_tokens'])
    return df[RESULT_COLS]

def summarize_target(target):
    """Aggregate this target's records on disk, by model."""
    df = pd.DataFrame(load_records(RECORDS_DIR, target=target))
    if not len(df):
        print(f'No records for {target} yet.')
        return None
    df['billed_input_tokens'] = billed_input_tokens(df['input_tokens'], df['cache_read_tokens'], df['cache_creation_tokens'])
    return df.groupby('model').agg(
        n=('query_id', 'count'), passed=('validation_passed', 'sum'),
        mean_iterations=('iterations_used', 'mean'),
        in_tok=('billed_input_tokens', 'sum'), out_tok=('output_tokens', 'sum'),
        mean_duration_s=('duration_seconds', 'mean'))

print(f'SUBSET = {SUBSET}')

SUBSET = None


## SQL → Cypher

### llama3.2:latest

In [ ]:
run_cell('cypher', 'llama3.2:latest')

### qwen3-coder:30b

In [ ]:
run_cell('cypher', 'qwen3-coder:30b')

### gemma4:26b

In [ ]:
run_cell('cypher', 'gemma4:26b')

### claude-opus-4-8

In [ ]:
run_cell('cypher', 'claude-opus-4-8')

### Aggregation by model

In [ ]:
summarize_target('cypher')

## SQL → AQL

### llama3.2:latest

In [ ]:
run_cell('aql', 'llama3.2:latest')

### qwen3-coder:30b

In [ ]:
run_cell('aql', 'qwen3-coder:30b')

### gemma4:26b

In [ ]:
run_cell('aql', 'gemma4:26b')

### claude-opus-4-8

In [ ]:
run_cell('aql', 'claude-opus-4-8')

### Aggregation by model

In [ ]:
summarize_target('aql')

## SQL → Gremlin

### llama3.2:latest

In [ ]:
run_cell('gremlin', 'llama3.2:latest')

### qwen3-coder:30b

In [ ]:
run_cell('gremlin', 'qwen3-coder:30b')

### gemma4:26b

In [ ]:
run_cell('gremlin', 'gemma4:26b')

### claude-opus-4-8

In [ ]:
run_cell('gremlin', 'claude-opus-4-8')

### Aggregation by model

In [ ]:
summarize_target('gremlin')

## Run-level summary

In [3]:
import pandas as pd

df = pd.DataFrame(load_records(RECORDS_DIR))
if len(df):
    df['billed_input_tokens'] = billed_input_tokens(df['input_tokens'], df['cache_read_tokens'], df['cache_creation_tokens'])
    print(f'Total records: {len(df)}')
    print(f"Validation passed: {int(df['validation_passed'].sum())} / {len(df)}")
    print(f"Total tokens: {int(df['billed_input_tokens'].sum()):,} in / {int(df['output_tokens'].sum()):,} out")
    print(f"Mean duration: {df['duration_seconds'].mean():.2f}s")
    display(df.groupby(['dataset','target','model']).agg(
        n=('query_id','count'), passed=('validation_passed','sum'),
        mean_iter=('iterations_used','mean'),
        in_tok=('billed_input_tokens','sum'), out_tok=('output_tokens','sum')))
else:
    print('No records yet.')

Total records: 168
Validation passed: 152 / 168
Total tokens: 815,607 in / 135,310 out
Mean duration: 15.58s


n  passed  mean_iter  in_tok  out_tok
dataset target  model                                                  
ldbc    aql     claude-opus-4-8  14      14   1.000000   84841     1802
                gemma4:26b       14      14   1.071429   61532    43420
                llama3.2:latest  14      12   1.714286   96810     4716
                qwen3-coder:30b  14      14   1.000000   51364     1049
        cypher  claude-opus-4-8  14      14   1.000000   66040     1329
                gemma4:26b       14      14   1.000000   46127    23720
                llama3.2:latest  14      14   1.285714   53254     1955
                qwen3-coder:30b  14      14   1.000000   40910      833
        gremlin claude-opus-4-8  14      14   1.000000   68470     1341
                gemma4:26b       14      14   1.071429   53359    44454
                llama3.2:latest  14       3   2.571429  127989     8685
                qwen3-coder:30b  14      11   1.428571   64911     2006

Records written. Proceed to `02_behavioural_metrics.ipynb`.